# 10_tree_queue_pipeline——章节测试

请先完成 `10.01` 和 `10.02`，再独立回答以下问题。

## 一、选择题

### 题目 1：`parent[i] = p` 表示什么？根节点为什么使用 `-1`？

A. 节点 `p` 的父节点是 `i`；`-1` 表示根节点没有父节点  B. 节点 `i` 的父节点是 `p`；`-1` 表示根节点没有父节点  C. 节点 `i` 的父节点是 `p`；`-1` 表示根节点有多个父节点  D. 节点 `p` 的父节点是 `i`；`-1` 表示循环依赖

In [ ]:
answer1 = None  # 请填写 A/B/C/D
print(f"题目1：{answer1}")

### 题目 2：`TreeQueuePipelineLite` 为什么使用单个 control block 而不是多核并行？

A. 910B 只有一个 AI Core  B. 调度含跨任务依赖，阶段时序需要递推保持确定性  C. 多核会导致 FP16 精度下降  D. 使用多个 control block 会触发 workspace 写入错误

In [ ]:
answer2 = None  # 请填写 A/B/C/D
print(f"题目2：{answer2}")

### 题目 3：`queue_depth=2` 的双缓冲带来的主要收益是什么？

A. 减少任务数量，缩短树的层数  B. 使下一次 `CopyIn` 与上一任务的 `Compute`/`CopyOut` 发生重叠  C. 提高 `stage_end` 的 FP16 精度  D. 让优先队列可以违反父子依赖

In [ ]:
answer3 = None  # 请填写 A/B/C/D
print(f"题目3：{answer3}")

## 二、填空题

1. BFS frontier 必须使用 ______ 队列才能保持层序遍历（填 FIFO 或 LIFO）。
2. 910B 构建命令使用 `TARGET=________`。
3. Kernel 通过 tiling 数据读取 `taskCount`、`queueDepth` 和 ______，不在代码中写死树规模和层数。

In [ ]:
answer4 = None
answer5 = None
answer6 = None
print(answer4, answer5, answer6)

## 三、实践任务

### 任务 1：阅读并解释调度 Kernel

阅读 `src/tree_queue_lab/custom_ops/src/TreeQueuePipelineLite/op_kernel/tree_queue_pipeline_lite.cpp` 的 `Process()`，标出：任务合法性与父子约束检查、缓冲槽选择、Compute lane 选择、`stage_end` 写回四个阶段，并用一段 Markdown 说明为什么父子检查采用在线扫描而不是位置表。

### 任务 2：完成 910B 构建配置检查

说明 `TARGET=ascend910b` 如何传给 `build_ops.sh` 并被 `msopgen` 使用，并指出 Kernel 为什么不需要针对 910B 修改（tiling 中的哪些字段负责适配）。

### 任务 3：解释双缓冲流水线的时间构成

运行 `python3 scripts/run_lab.py --data_dir data --output data/output.json`，把 `queue_depth` 分别改为 1、2、3 后记录 `end_to_end`，解释为什么队列深度增大到一定程度后收益递减。

In [ ]:
from pathlib import Path
import json
import sys
import subprocess

chapter_dir = Path.cwd()
while not (chapter_dir / 'src' / 'tree_queue_lab').is_dir() and chapter_dir != chapter_dir.parent:
    chapter_dir = chapter_dir.parent
src_dir = chapter_dir / 'src' / 'tree_queue_lab'
sys.path.insert(0, str(src_dir / 'scripts'))
from scheduler import pipeline_schedule, priority_schedule

data_dir = src_dir / 'data'
if not (data_dir / 'input.json').is_file():
    subprocess.run([sys.executable, str(src_dir / 'scripts' / 'gen_data.py'), '--num_nodes', '31', '--seed', '10', '--output', str(data_dir)], check=True)
payload = json.loads((src_dir / 'data' / 'input.json').read_text(encoding='utf-8'))
order = priority_schedule(payload['parent'], payload['cost'])
results = {}
for depth in (1, 2, 3):
    timing = pipeline_schedule(
        order, payload['cost'], depth, payload['copy_in'],
        payload['copy_out'], payload['compute_lanes']
    )
    results[depth] = timing['end_to_end']
    assert all(event['copy_in'][1] <= event['compute'][0] <= event['compute'][1] <= event['copy_out'][0] for event in timing['events'])
    print(f'queue_depth={depth}: end_to_end={timing["end_to_end"]:.1f}')
assert results[1] >= results[2] >= results[3]
print('实践通过：阶段顺序和队列深度比较结果满足资源约束。')

完成全部题目后，运行下一个代码单元查看实践解析。

In [ ]:
!cat answer/tree_queue_lab_answer.md